In [22]:

import numpy as np
import joblib
import torch
import os 
import csv
import torch.nn as nn
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from skimage.filters import threshold_otsu

REF_ARTICULO_CAE = {1: (0.92, 0.71), 5: (0.90, 0.90), 10: (0.88, 0.92), 15: (0.81, 0.84),
                    20: (0.71, 0.77), 25: (0.67, 0.73), 30: (0.59, 0.71)}
REF_ARTICULO_CAE_OTSU = {1: (0.62, 0.95), 5: (0.62, 0.97), 10: (0.59, 0.98), 15: (0.53, 0.94),
                    20: (0.44, 0.92), 25: (0.42, 0.89), 30: (0.31, 0.86)}
REF_ARTICULO_RF = {1: (0.95, 0.56), 5:(0.94, 0.86), 10:(0.90, 0.91), 15:(0.87, 0.90), 
                 20:(0.83, 0.89), 25:(0.73, 0.85), 30:(0.60, 0.77)}

UMBRAL = 0.75
NIVELES = (1, 5, 10, 15, 20, 25, 30)
DIR_EXPERIMENTO = (
    "/home/angelo/MyStore/Proyecto Investigador/2017/entrenamiento train fiables test/"
)

In [2]:
class Conv1DAutoencoder(nn.Module):
    def __init__(self, input_channels=2, seq_len=61, bottleneck_dim=1):
        super().__init__()

        self.input_channels = input_channels
        self.seq_len = seq_len

        # ============================================================
        # ENCODER CONVOLUCIONAL
        # ============================================================
        self.encoder = nn.Sequential(
            nn.Conv1d(input_channels, 64, kernel_size=7, stride=1, padding=1),
            nn.ELU(),
            nn.MaxPool1d(kernel_size=2, stride=2),  # 61 -> 28

            nn.Conv1d(64, 128, kernel_size=5, stride=1, padding=0),
            nn.ELU(),
            nn.MaxPool1d(kernel_size=2, stride=2),  # 28 -> 12

            nn.Conv1d(128, 256, kernel_size=3, stride=1, padding=0),
            nn.ELU(),
            nn.MaxPool1d(kernel_size=2, stride=2),  # 12 -> 5
        )

        self.flatten = nn.Flatten()

        # ============================================================
        # FC ENCODER (Bottleneck)
        # ============================================================
        self.fc_enc = nn.Sequential(
            nn.Linear(256 * 5, 128),
            nn.ELU(),
            nn.Linear(128, 64),
            nn.ELU(),
            nn.Linear(64, 32),
            nn.ELU(),
            nn.Linear(32, bottleneck_dim),
            nn.ELU()  # Activación ELU en el espacio latente (Tabla 2 del paper)
        )

        # ============================================================
        # FC DECODER (Adaptable dinámicamente)
        # ============================================================
        self.fc_dec = nn.Sequential(
            nn.Linear(bottleneck_dim, 32),
            nn.ELU(),
            nn.Linear(32, 64),
            nn.ELU(),
            nn.Linear(64, 128),
            nn.ELU(),
            nn.Linear(128, input_channels * seq_len),  # 2x61=122 o 4x61=244
        )

    def forward(self, x):
        x_enc = self.encoder(x)                            # (B, 256, 5)
        x_flat = self.flatten(x_enc)                       # (B, 1280)
        z = self.fc_enc(x_flat)                            # (B, bottleneck_dim)
        x_dec = self.fc_dec(z)                             # (B, C * T)
        x_dec = x_dec.view(-1, self.input_channels, self.seq_len) # (B, C, T)
        return x_dec

In [ ]:
"""
Evaluación del CAE y del RF (sin Otsu) sobre los conjuntos infectados.

Requiere definidos fuera de este bloque:
    Conv1DAutoencoder, REF_ARTICULO_CAE, REF_ARTICULO_RF
"""

# ==========================================================
# CAE
# ==========================================================
def cargar_modelos(path_modelos, clases, pasada=10, input_channels=2, seq_len=61,
                   bottleneck_dim=1, device="cpu", infestado=False):
    """Carga los CAE guardados (normales o infestados `_crop_`)."""
    modelos = {}
    for c in clases:
        nombre = (f"cae_pasada_{pasada}_crop_{c}.pth" if infestado
                  else f"cae_pasada_{pasada}_{c}.pth")
        ruta = os.path.join(path_modelos, nombre)
        if not os.path.exists(ruta):
            print(f"Sin modelo para {c}: {ruta}")
            continue
        checkpoint = torch.load(ruta, map_location=device, weights_only=False)
        m = Conv1DAutoencoder(input_channels=input_channels, seq_len=seq_len,
                              bottleneck_dim=bottleneck_dim).to(device)
        m.load_state_dict(checkpoint["model_state_dict"])
        m.eval()
        modelos[c] = m
    return modelos


def predict_pixels(datos_pixeles, etiquetas_reales, modelos_cae, batch_size=4096, device="cuda"):
    """
    Para cada píxel calcula el MSE con cada CAE experto y toma como clase
    candidata la del CAE con menor error.
    """
    if isinstance(datos_pixeles, np.ndarray):
        datos_pixeles = torch.as_tensor(datos_pixeles, dtype=torch.float32)
    if isinstance(etiquetas_reales, torch.Tensor):
        etiquetas_reales = etiquetas_reales.cpu().numpy()
    elif isinstance(etiquetas_reales, list):
        etiquetas_reales = np.array(etiquetas_reales)

    n = datos_pixeles.shape[0]
    clases_modelos = sorted(modelos_cae.keys())
    matriz_errores = np.zeros((n, len(clases_modelos)), dtype=np.float32)
    funcion_mse = nn.MSELoss(reduction="none")

    for j, clase in enumerate(clases_modelos):
        modelo = modelos_cae[clase].to(device)
        modelo.eval()
        errores = []
        with torch.no_grad():
            for ini in range(0, n, batch_size):
                lote = datos_pixeles[ini:ini + batch_size].to(device)
                errores.append(funcion_mse(lote, modelo(lote)).mean(dim=(1, 2)).cpu().numpy())
        matriz_errores[:, j] = np.concatenate(errores)

    candidatas = np.array(clases_modelos)[np.argmin(matriz_errores, axis=1)]
    fiables = candidatas == etiquetas_reales
    return {
        "mse_matrix": matriz_errores,
        "candidate_classes": candidatas,
        "is_trustworthy": fiables,
        "is_suspicious": ~fiables,
    }


# ==========================================================
# REGLA DE RELABEL Y DIAGNÓSTICO (comunes a CAE y RF)
# ==========================================================
def relabel_por_parcela(candidatas, y_inf, pid, umbral=UMBRAL):
    """
    Una parcela se relabela si más del `umbral` de sus píxeles comparten la misma
    clase candidata y esa clase difiere de su etiqueta (infectada).
    Devuelve: ids de parcela, etiqueta infectada, etiqueta final, máscara relabel.
    """
    upid, first, inv = np.unique(pid, return_index=True, return_inverse=True)
    etiqueta = y_inf[first]
    clases, cinv = np.unique(candidatas, return_inverse=True)
    M = np.zeros((len(upid), len(clases)))
    np.add.at(M, (inv, cinv), 1)
    frac = M / M.sum(1, keepdims=True)
    nueva = clases[frac.argmax(1)]
    relabel = (frac.max(1) > umbral) & (nueva != etiqueta)
    return upid, etiqueta, np.where(relabel, nueva, etiqueta), relabel


def diagnostico(orig, inf, final, relabel):
    infectada = inf != orig
    return {
        "infectadas": int(infectada.sum()),
        "relabels": int(relabel.sum()),
        "acierto": int((relabel & infectada & (final == orig)).sum()),       # devuelta a su clase original
        "mal_destino": int((relabel & infectada & (final != orig)).sum()),   # detectada, pero a otra clase
        "no_detectada": int((infectada & ~relabel).sum()),                   # infectada sin relabel
        "falso_positivo": int((relabel & ~infectada).sum()),                 # parcela limpia relabelada
    }
    
def _fila_metricas(orig, etiq_inf, final, relabel, pct, rep, n_parcelas):
    d = diagnostico(orig, etiq_inf, final, relabel)
    d.update(recall=d["acierto"] / max(d["infectadas"], 1),
             precision=d["acierto"] / max(d["relabels"], 1),
             pct=pct, rep=rep, n_parcelas=int(n_parcelas))
    return d

def _guardar_csv(filas, ruta_csv):
    if ruta_csv and filas:
        with open(ruta_csv, "w", newline="") as f:
            w = csv.DictWriter(f, fieldnames=list(filas[0].keys()))
            w.writeheader()
            w.writerows(filas)
        print(f"Guardado: {ruta_csv}")
        
def _cargar_replica(pct, rep, dir_datasets):
    ruta_pt = os.path.join(dir_datasets, f"dataset_noise_{pct:02d}pct_rep_{rep:02d}.pt")
    data = torch.load(ruta_pt, map_location="cpu", weights_only=False)
    return data["train"], data["meta"]


def _orig_desde_meta(meta, upid, etiq_inf):
    inf_map = dict(zip(meta["ids_fiables"], meta["etiqueta_infectada"]))
    orig_map = dict(zip(meta["ids_fiables"], meta["clase_fiable"]))
    assert all(str(inf_map[p]) == str(e) for p, e in zip(upid, etiq_inf)), \
        "Las etiquetas del dataset no coinciden con meta['etiqueta_infectada']"
    return np.array([orig_map[p] for p in upid])


# ==========================================================
# EVALUACIÓN CAE
# ==========================================================
def evaluar_replica(pct, rep, dir_datasets, dir_modelos, device, umbral=UMBRAL, pasada=10,
                    dir_parcelas=None):
    datos, meta = _cargar_replica(pct, rep, dir_datasets)
    clases = sorted(datos.keys())

    X = torch.cat([torch.as_tensor(datos[c]["X"], dtype=torch.float32).reshape(-1, 2, 61)
                   for c in clases])
    y_inf = np.concatenate([np.full(len(datos[c]["X"]), c) for c in clases])
    pid = np.concatenate([np.asarray(datos[c]["parcelas"]) for c in clases])

    dir_rep = os.path.join(dir_modelos, f"ruido_{pct:02d}pct", f"rep_{rep:02d}")
    modelos = cargar_modelos(dir_rep, clases, pasada=pasada, device=device, infestado=True)
    assert len(modelos) > 0, f"No se cargó ningún modelo desde {dir_rep}"

    cand = predict_pixels(X, y_inf, modelos, batch_size=4096, device=device)["candidate_classes"]
    upid, etiq_inf, final, relabel = relabel_por_parcela(cand, y_inf, pid, umbral)
    orig = _orig_desde_meta(meta, upid, etiq_inf)

    if dir_parcelas is not None:
        os.makedirs(dir_parcelas, exist_ok=True)
        np.savez(os.path.join(dir_parcelas, f"cae_{pct:02d}pct_rep_{rep:02d}.npz"),
                 upid=upid, orig=orig, inf=etiq_inf, final=final, relabel=relabel)

    return _fila_metricas(orig, etiq_inf, final, relabel, pct, rep, len(upid))


def evaluar_nivel(pct, n_rep, dir_datasets, dir_modelos, device, umbral=UMBRAL):
    filas = []
    print(f"{'rep':>3} {'parc':>5} {'infect':>6} {'relab':>5} {'ok':>3} {'malDest':>7} "
          f"{'noDet':>5} {'FP':>4} {'recall':>7} {'prec':>6}")
    for rep in range(n_rep):
        try:
            d = evaluar_replica(pct, rep, dir_datasets, dir_modelos, device, umbral)
        except (FileNotFoundError, AssertionError) as e:
            print(f"{rep:>3}  ERROR: {e}")
            continue
        filas.append(d)
        print(f"{rep:>3} {d['n_parcelas']:>5} {d['infectadas']:>6} {d['relabels']:>5} "
              f"{d['acierto']:>3} {d['mal_destino']:>7} {d['no_detectada']:>5} "
              f"{d['falso_positivo']:>4} {d['recall']:>7.2f} {d['precision']:>6.2f}")
    if filas:
        rec = np.array([f["recall"] for f in filas])
        pre = np.array([f["precision"] for f in filas])
        ok = sum(f["acierto"] for f in filas)
        print(f"\nMedia ± std   recall {rec.mean():.2f} ± {rec.std():.2f}   "
              f"precisión {pre.mean():.2f} ± {pre.std():.2f}")
        print(f"Agregado ({len(filas)} réplicas)   "
              f"recall {ok / max(sum(f['infectadas'] for f in filas), 1):.2f}   "
              f"precisión {ok / max(sum(f['relabels'] for f in filas), 1):.2f}")
    return filas


def resumen_niveles(filas):
    print(f"\n{'ruido':>5} {'n':>3} | {'recall':>11} {'precisión':>11} | "
          f"{'recall agr':>10} {'prec. agr':>9} | {'art. rec':>8} {'art. prec':>9}")
    for pct in sorted({f["pct"] for f in filas}):
        fs = [f for f in filas if f["pct"] == pct]
        rec = np.array([f["recall"] for f in fs])
        pre = np.array([f["precision"] for f in fs])
        ok = sum(f["acierto"] for f in fs)
        rec_agr = ok / max(sum(f["infectadas"] for f in fs), 1)
        pre_agr = ok / max(sum(f["relabels"] for f in fs), 1)
        ref = REF_ARTICULO_CAE.get(pct, (float("nan"), float("nan")))
        print(f"{pct:>4}% {len(fs):>3} | {rec.mean():.2f}±{rec.std():.2f}   {pre.mean():.2f}±{pre.std():.2f}   | "
              f"{rec_agr:>10.2f} {pre_agr:>9.2f} | {ref[0]:>8.2f} {ref[1]:>9.2f}")


def evaluar_todos(dir_datasets, dir_modelos, device, niveles=NIVELES, n_rep=10,
                  umbral=UMBRAL, ruta_csv=None, dir_parcelas=None):
    filas = []
    for pct in niveles:
        for rep in range(n_rep):
            try:
                d = evaluar_replica(pct, rep, dir_datasets, dir_modelos, device, umbral,
                                    dir_parcelas=dir_parcelas)
            except (FileNotFoundError, AssertionError) as e:
                print(f"[CAE {pct}% rep {rep}] omitida: {e}")
                continue
            filas.append(d)
            if str(device).startswith("cuda"):
                torch.cuda.empty_cache()
        print(f"Nivel {pct}% evaluado")

    _guardar_csv(filas, ruta_csv)
    if filas:
        resumen_niveles(filas)
    return filas


# ==========================================================
# EVALUACIÓN RF (reentrenado por réplica con etiquetas infectadas)
# ==========================================================
def cargar_params_rf(ruta_rf):
    assert ruta_rf is not None, "Falta RUTA_RF"
    contenido = joblib.load(ruta_rf)
    rf = contenido["model"] if isinstance(contenido, dict) else contenido
    return {k: v for k, v in rf.get_params().items() if k != "random_state"}

def _limitar_por_parcela(idx, pid, max_pix, rng):
    """De los índices idx, conserva como mucho max_pix píxeles al azar por parcela."""
    idx = idx[rng.permutation(len(idx))]
    _, inv = np.unique(pid[idx], return_inverse=True)
    orden = np.argsort(inv, kind="stable")
    inv_ord = inv[orden]
    inicio = np.r_[0, np.flatnonzero(np.diff(inv_ord)) + 1]
    rango = np.arange(len(inv_ord)) - inicio[inv_ord]
    return idx[orden[rango < max_pix]]


def predecir_oof_rf(X, y_inf, pid, semilla, rf_params, agrupar=True, max_pix_parcela=100, n_splits=4):
    """Predicción fuera de fold sobre el dataset infectado (4-fold CV, como en el artículo)."""
    oof = np.empty(len(y_inf), dtype=y_inf.dtype)

    if agrupar:  # folds por parcela: el RF nunca ve píxeles de la parcela que predice
        splits = StratifiedGroupKFold(n_splits, shuffle=True, random_state=semilla).split(X, y_inf, groups=pid)
    else:        # folds por píxel
        splits = StratifiedKFold(n_splits, shuffle=True, random_state=semilla).split(X, y_inf)

    for k, (tr, te) in enumerate(splits):
        if max_pix_parcela:
            tr = _limitar_por_parcela(tr, pid, max_pix_parcela, np.random.default_rng(semilla + k))
        rf = RandomForestClassifier(**rf_params, random_state=semilla)
        rf.fit(X[tr], y_inf[tr])
        oof[te] = rf.predict(X[te])  # se predicen TODOS los píxeles del fold
    return oof


def evaluar_replica_rf(pct, rep, dir_datasets, rf_params, umbral=UMBRAL, agrupar=True,
                       max_pix_parcela=100, dir_parcelas=None):
    datos, meta = _cargar_replica(pct, rep, dir_datasets)
    clases = sorted(datos.keys())

    # X aplanado a 122 valores (2 canales x 61 fechas)
    X = np.concatenate([torch.as_tensor(datos[c]["X"], dtype=torch.float32)
                        .reshape(len(datos[c]["X"]), -1).numpy() for c in clases])
    y_inf = np.concatenate([np.full(len(datos[c]["X"]), c) for c in clases])
    pid = np.concatenate([np.asarray(datos[c]["parcelas"]) for c in clases])

    oof = predecir_oof_rf(X, y_inf, pid, semilla=1000 * pct + rep, rf_params=rf_params,
                          agrupar=agrupar, max_pix_parcela=max_pix_parcela)

    upid, etiq_inf, final, relabel = relabel_por_parcela(oof, y_inf, pid, umbral)
    orig = _orig_desde_meta(meta, upid, etiq_inf)

    if dir_parcelas is not None:
        os.makedirs(dir_parcelas, exist_ok=True)
        np.savez(os.path.join(dir_parcelas, f"rf_{pct:02d}pct_rep_{rep:02d}.npz"),
                 upid=upid, orig=orig, inf=etiq_inf, final=final, relabel=relabel)

    return _fila_metricas(orig, etiq_inf, final, relabel, pct, rep, len(upid))


def evaluar_todos_rf(dir_datasets, ruta_rf, niveles=NIVELES, n_rep=10, umbral=UMBRAL,
                     agrupar=True, max_pix_parcela=100, ruta_csv=None, dir_parcelas=None):
    rf_params = cargar_params_rf(ruta_rf)   # una sola carga para las 70 réplicas
    filas = []
    for pct in niveles:
        for rep in range(n_rep):
            try:
                d = evaluar_replica_rf(pct, rep, dir_datasets, rf_params, umbral, agrupar,
                                       max_pix_parcela, dir_parcelas)
            except (FileNotFoundError, AssertionError) as e:
                print(f"[RF {pct}% rep {rep}] omitida: {e}")
                continue
            filas.append(d)
        print(f"RF nivel {pct}% evaluado")

    _guardar_csv(filas, ruta_csv)
    return filas


# ==========================================================
# RECONSTRUIR FILAS DESDE .npz YA GUARDADOS (sin recalcular)
# ==========================================================
def filas_desde_npz(dir_base, prefijo=None, niveles=NIVELES, n_rep=10):
    """
    - Formato plano (CAE/RF sklearn):  dir_base/{prefijo}_{pct:02d}pct_rep_{rep:02d}.npz
    - Formato carpetas (RF Spark):     dir_base/ruido_{pct:02d}pct/rep_{rep:02d}/resultado_parcelas.npz
      (usar prefijo=None)
    """
    filas = []
    for pct in niveles:
        for rep in range(n_rep):
            if prefijo is None:
                ruta = os.path.join(dir_base, f"ruido_{pct:02d}pct", f"rep_{rep:02d}",
                                    "resultado_parcelas.npz")
            else:
                ruta = os.path.join(dir_base, f"{prefijo}_{pct:02d}pct_rep_{rep:02d}.npz")
            if not os.path.exists(ruta):
                print(f"[{prefijo or 'rf_spark'} {pct}% rep {rep}] no existe")
                continue
            z = np.load(ruta, allow_pickle=True)
            filas.append(_fila_metricas(z["orig"], z["inf"], z["final"], z["relabel"],
                                        pct, rep, len(z["upid"])))
    return filas


# ==========================================================
# TABLA COMPARATIVA CAE vs RF vs ARTÍCULO
# ==========================================================
def resumen_comparativo(filas_cae, filas_rf):
    def est(filas, pct):
        fs = [f for f in filas if f["pct"] == pct]
        if not fs:
            return ("--", "--")
        rec = np.array([f["recall"] for f in fs])
        pre = np.array([f["precision"] for f in fs])
        return (f"{rec.mean():.2f}±{rec.std():.2f}", f"{pre.mean():.2f}±{pre.std():.2f}")

    def fmt_ref(ref):
        return f"{ref[0]:.2f}/{ref[1]:.2f}" if not np.isnan(ref[0]) else "--/--"

    print(f"\n{'ruido':>5} | {'--- CAE (tuyo vs artículo) ---':^33} | {'--- RF (tuyo vs artículo) ---':^33}")
    print(f"{'':>5} | {'CAE rec':>10} {'CAE prec':>10} {'Art r/p':>10} | {'RF rec':>10} {'RF prec':>10} {'Art r/p':>10}")
    print("-" * 83)

    for pct in sorted({f["pct"] for f in filas_cae} | {f["pct"] for f in filas_rf}):
        c = est(filas_cae, pct)
        r = est(filas_rf, pct)
        ac = REF_ARTICULO_CAE.get(pct, (np.nan, np.nan))
        ar = REF_ARTICULO_RF.get(pct, (np.nan, np.nan))
        print(f"{pct:>4}% | {c[0]:>10} {c[1]:>10} {fmt_ref(ac):>10} | "
              f"{r[0]:>10} {r[1]:>10} {fmt_ref(ar):>10}")


# ==========================================================
# USO
# ==========================================================
# filas_cae = evaluar_todos(dir_datasets_A, dir_modelos_A, device, dir_parcelas=dir_parcelas)
# filas_rf  = evaluar_todos_rf(dir_datasets_A, RUTA_RF, dir_parcelas=dir_parcelas)
# resumen_comparativo(filas_cae, filas_rf)
#
# # o, si ya tienes los .npz:
# filas_cae = filas_desde_npz(dir_parcelas, "cae")
# filas_rf  = filas_desde_npz(dir_parcelas, "rf")
# resumen_comparativo(filas_cae, filas_rf)

In [ ]:
dir_base_modelos = DIR_EXPERIMENTO + "modelos_experimento_farmsar_test_v1/"
dir_base_datasets = DIR_EXPERIMENTO + "dataset_ruidoso_farmsar_test_v1/"
device = 'cuda' if torch.cuda.is_available() else 'cpu'

res_1 = evaluar_nivel(1, 10, dir_base_datasets, dir_base_modelos, device)

rep  parc infect relab  ok malDest noDet   FP  recall   prec
  0  1216     10    14  10       0     0    4    1.00   0.71
  1  1216     12    39  11       0     1   28    0.92   0.28
  2  1216     12    17  11       0     1    6    0.92   0.65
  3  1216      6    13   6       0     0    7    1.00   0.46
  4  1216     10    14  10       0     0    4    1.00   0.71
  5  1216     12    57   8       2     2   47    0.67   0.14
  6  1216     10    14   8       0     2    6    0.80   0.57
  7  1216      8    13   8       0     0    5    1.00   0.62
  8  1216     12    21  12       0     0    9    1.00   0.57
  9  1216     10    17   9       0     1    8    0.90   0.53

Media ± std   recall 0.92 ± 0.11   precisión 0.52 ± 0.18
Agregado (10 réplicas)   recall 0.91   precisión 0.42


In [7]:
# ==========================================
# 1. CONFIGURACIÓN DE RUTAS Y DISPOSITIVO
# ==========================================
dir_base_datasets = DIR_EXPERIMENTO + "dataset_ruidoso_farmsar_test_v1/"
dir_base_modelos  = DIR_EXPERIMENTO + "modelos_experimento_farmsar_test_v1/"
dir_parcelas_cae  = DIR_EXPERIMENTO + "resultados_CAE_parcelas_v1/"
dir_parcelas_rf = DIR_EXPERIMENTO + "resultados_RF_backscatter_v1/"

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# ==========================================
# 2. EVALUAR MODELO CAE (Ya lo tenías)
# ==========================================
print("Evaluando modelos CAE...")
filas_CAE = evaluar_todos(
    dir_base_datasets, 
    dir_base_modelos, 
    device, 
    niveles=(1, 5, 10, 15, 20, 25, 30),
    n_rep=10, 
    umbral=0.75, 
    ruta_csv="resultados_cae.csv", 
    dir_parcelas=dir_parcelas_cae
)
filas_rf  = filas_desde_npz(dir_parcelas_rf)
resumen_comparativo(filas_CAE, filas_rf)

Evaluando modelos CAE...
Nivel 1% evaluado
Sin modelo para Fallow: /home/angelo/MyStore/Proyecto Investigador/2017/entrenamiento train fiables test/modelos_experimento_farmsar_test_v1/ruido_05pct/rep_01/cae_pasada_10_crop_Fallow.pth
Sin modelo para Fallow: /home/angelo/MyStore/Proyecto Investigador/2017/entrenamiento train fiables test/modelos_experimento_farmsar_test_v1/ruido_05pct/rep_02/cae_pasada_10_crop_Fallow.pth
Sin modelo para Fallow: /home/angelo/MyStore/Proyecto Investigador/2017/entrenamiento train fiables test/modelos_experimento_farmsar_test_v1/ruido_05pct/rep_05/cae_pasada_10_crop_Fallow.pth
Sin modelo para Fallow: /home/angelo/MyStore/Proyecto Investigador/2017/entrenamiento train fiables test/modelos_experimento_farmsar_test_v1/ruido_05pct/rep_07/cae_pasada_10_crop_Fallow.pth
Sin modelo para Alfalfa: /home/angelo/MyStore/Proyecto Investigador/2017/entrenamiento train fiables test/modelos_experimento_farmsar_test_v1/ruido_05pct/rep_08/cae_pasada_10_crop_Alfalfa.pth
Sin m

In [37]:
"""
Comparación CAE vs RF, SIN y CON Otsu (autocontenido).

Este script ya incluye todo lo relacionado con Otsu. Solo necesita que en el kernel
existan las funciones que ya tienes de antes:
    cargar_modelos, predict_pixels, relabel_por_parcela, diagnostico, Conv1DAutoencoder
y las referencias del artículo:
    REF_ARTICULO_CAE   (CAE sin Otsu)   y   REF_ARTICULO_RF

Además, los .npz del RF deben contener `prob_media` y `clases`
(los genera la versión nueva de farmsar_rf_baseline.py).
"""

# ==========================================================
# 1. CONFIGURACIÓN
# ==========================================================

dir_datasets = DIR_EXPERIMENTO + "dataset_ruidoso_farmsar_test_v1/"
dir_modelos = DIR_EXPERIMENTO + "modelos_experimento_farmsar_test_v1/"
dir_parcelas_cae = DIR_EXPERIMENTO + "resultados_CAE_parcelas_v1/"
dir_parcelas_rf = DIR_EXPERIMENTO + "resultados_RF_backscatter_v1/"
ruta_csv_salida = DIR_EXPERIMENTO + "comparacion_cae_rf_otsu_v1.csv"

device = "cuda" if torch.cuda.is_available() else "cpu"
NIVELES_CMP = (1, 5, 10, 15, 20, 25, 30)
N_REP_CMP = 10
UMBRAL_CMP = 0.75

def _comprobar_dependencias():
    necesarias = ["cargar_modelos", "predict_pixels", "relabel_por_parcela", "diagnostico",
              "Conv1DAutoencoder", "REF_ARTICULO_CAE", "REF_ARTICULO_RF","REF_ARTICULO_CAE_OTSU"]
    faltan = [n for n in necesarias if n not in globals()]
    assert not faltan, f"Falta definir antes de usar este script: {faltan}"

def aplicar_margen_mse(err_par, clases_cols, etiq_inf, final, relabel,
                       margen_base=0.20, margen_2do=0.30):
    """
    Alternativa a Otsu basada en margen relativo MSE.

    Relabela la parcela i solo si el CAE de su clase original
    reconstruye significativamente peor que el CAE candidato:

        (MSE_orig - MSE_cand) / MSE_cand > umbral

    Umbral:
        - 30% si la clase original es la 2ª mejor (caso ambiguo)
        - 20% en el resto de casos
    """
    col    = {c: j for j, c in enumerate(clases_cols)}
    rel    = relabel.copy()

    for i in np.flatnonzero(relabel):

        o = etiq_inf[i]          # clase original (infectada)

        if o not in col:
            rel[i] = False
            continue

        # Ordenar clases por MSE ascendente para esta parcela
        orden    = np.argsort(err_par[i])
        clase_2  = clases_cols[orden[1]]   # segunda mejor

        mse_cand = err_par[i, orden[0]]    # MSE del candidato
        mse_orig = err_par[i, col[o]]      # MSE de la clase original

        # Diferencia relativa: cuánto peor reconstruye la clase original
        diferencia = (mse_orig - mse_cand) / mse_cand

        # Caso ambiguo: clase original es la 2ª mejor → más conservador
        if clase_2 == o:
            rel[i] = diferencia > margen_2do
        else:
            rel[i] = diferencia > margen_base

    return rel

# ==========================================================
# 2. NÚCLEO OTSU
# ==========================================================
def _error_por_parcela(M_pixel, inv, n_parc):
    """Media por parcela de cada columna de M_pixel (n_pix x n_clases) -> (n_parc, n_clases)."""
    cnt = np.bincount(inv, minlength=n_parc)
    return np.stack([np.bincount(inv, weights=M_pixel[:, j], minlength=n_parc) / cnt
                     for j in range(M_pixel.shape[1])], axis=1)


def aplicar_otsu(err_par, clases_cols, etiq_inf, final, relabel):
    """
    err_par[i, j]: error de la parcela i bajo la clase clases_cols[j]
                   (CAE: MSE medio; RF: 1 - P(clase) medio).
    tau[c]: Otsu sobre las parcelas etiquetadas como c en el conjunto infectado.
    Doble condición (art. 3.4): la etiqueta cuestionada la explica mal (err > tau[o])
    y la clase nueva la explica bien (err < tau[n]).
    Sin tau para la clase origen o destino -> no se relabela (conservador).
    """
    col = {c: j for j, c in enumerate(clases_cols)}
    tau = {}
    for c, j in col.items():
        v = err_par[etiq_inf == c, j]
        if len(v) > 2 and np.ptp(v) > 0:
            tau[c] = threshold_otsu(v)

    rel = relabel.copy()
    for i in np.flatnonzero(relabel):
        o, n = etiq_inf[i], final[i]
        if o not in tau or n not in tau:
            rel[i] = False
        else:
            rel[i] = bool(err_par[i, col[o]] > tau[o] and err_par[i, col[n]] < tau[n])
    return rel, tau


def _filas_sin_con(orig, etiq_inf, final, relabel, relabel_otsu, tau, n_clases, pct, rep):
    n = len(orig)
    f_sin = _fila_metricas(orig, etiq_inf, final, relabel, pct, rep, n)
    f_con = _fila_metricas(orig, etiq_inf, final, relabel_otsu, pct, rep, n)
    for f in (f_sin, f_con):
        f["clases_sin_tau"] = n_clases - len(tau)   # clases a las que no se pudo calcular Otsu
    return f_sin, f_con


# ==========================================================
# 3. CAE: sin y con Otsu en una sola pasada
# ==========================================================
def evaluar_replica_cae_otsu(pct, rep, umbral=UMBRAL_CMP, pasada=10, dir_parcelas=None):

    # ── Cargar dataset infectado ──────────────────────────────
    ruta_pt = os.path.join(dir_datasets, f"dataset_noise_{pct:02d}pct_rep_{rep:02d}.pt")
    data = torch.load(ruta_pt, map_location="cpu", weights_only=False)
    datos, meta = data["train"], data["meta"]
    clases = sorted(datos.keys())

    # ── Construir X, etiquetas infectadas y parcelas ──────────
    X     = torch.cat([torch.as_tensor(datos[c]["X"], dtype=torch.float32)
                       .reshape(-1, 2, 61) for c in clases])
    y_inf = np.concatenate([np.full(len(datos[c]["X"]), c) for c in clases])
    pid   = np.concatenate([np.asarray(datos[c]["parcelas"]) for c in clases])

    # ── Cargar modelos CAE entrenados con datos infectados ────
    dir_rep = os.path.join(dir_modelos, f"ruido_{pct:02d}pct", f"rep_{rep:02d}")
    modelos = cargar_modelos(dir_rep, clases, pasada=pasada, device=device, infestado=True)
    assert len(modelos) > 0, f"No se cargó ningún modelo desde {dir_rep}"

    # ── Calcular errores MSE píxel a píxel ───────────────────
    res         = predict_pixels(X, y_inf, modelos, batch_size=4096, device=device)
    clases_cols = sorted(modelos.keys())

    # ── Regla del 75%: clase candidata por parcela ───────────
    upid, etiq_inf, final, relabel = relabel_por_parcela(
        res["candidate_classes"], y_inf, pid, umbral
    )

    # ── Recuperar etiquetas originales (antes de infectar) ───
    inf_map  = dict(zip(meta["ids_fiables"], meta["etiqueta_infectada"]))
    orig_map = dict(zip(meta["ids_fiables"], meta["clase_fiable"]))
    assert all(str(inf_map[p]) == str(e) for p, e in zip(upid, etiq_inf)), \
        "Las etiquetas del dataset no coinciden con meta['etiqueta_infectada']"
    orig = np.array([orig_map[p] for p in upid])

    # ── Error medio por parcela (necesario para Otsu y margen)
    _, inv  = np.unique(pid, return_inverse=True)
    err_par = _error_por_parcela(res["mse_matrix"], inv, len(upid))

    # ── Variante 1: sin umbral (solo regla del 75%) ───────────
    # relabel ya es esa variante, no hay que hacer nada más

    # ── Variante 2: Otsu doble condición (paper) ─────────────
    relabel_otsu, tau = aplicar_otsu(
        err_par, clases_cols, etiq_inf, final, relabel
    )

    # ── Variante 3: margen relativo MSE (propuesta propia) ───
    relabel_margen = aplicar_margen_mse(
        err_par, clases_cols, etiq_inf, final, relabel
    )

    # ── Calcular métricas para las tres variantes ─────────────
    n          = len(orig)
    f_sin      = _fila_metricas(orig, etiq_inf, final, relabel,        pct, rep, n)
    f_otsu     = _fila_metricas(orig, etiq_inf, final, relabel_otsu,   pct, rep, n)
    f_margen   = _fila_metricas(orig, etiq_inf, final, relabel_margen, pct, rep, n)

    # Añadimos info extra útil para el análisis
    clases_sin_modelo = len(clases) - len(modelos)
    clases_sin_tau    = len(clases_cols) - len(tau)

    for f in (f_sin, f_otsu, f_margen):
        f["clases_sin_modelo"] = clases_sin_modelo

    f_otsu["clases_sin_tau"]   = clases_sin_tau
    f_margen["clases_sin_tau"] = 0   # margen no necesita tau

    # ── Guardar resultados en disco si se pide ────────────────
    if dir_parcelas is not None:
        os.makedirs(dir_parcelas, exist_ok=True)
        np.savez(
            os.path.join(dir_parcelas, f"cae_otsu_{pct:02d}pct_rep_{rep:02d}.npz"),
            upid=upid, orig=orig, inf=etiq_inf, final=final,
            relabel=relabel, relabel_otsu=relabel_otsu, relabel_margen=relabel_margen
        )

    return f_sin, f_otsu, f_margen

def evaluar_todos_cae_otsu(niveles=NIVELES_CMP, n_rep=N_REP_CMP, umbral=UMBRAL_CMP,
                           dir_parcelas=None):
    sin, con, margen = [], [], []   # tres listas
    for pct in niveles:
        for rep in range(n_rep):
            try:
                f_sin, f_con, f_margen = evaluar_replica_cae_otsu(
                    pct, rep, umbral=umbral, pasada=10, dir_parcelas=dir_parcelas
                )
            except (FileNotFoundError, AssertionError) as e:
                print(f"[CAE {pct}% rep {rep}] omitida: {e}")
                continue
            sin.append(f_sin)
            con.append(f_con)
            margen.append(f_margen)   # guardamos la tercera variante 
            if str(device).startswith("cuda"):
                torch.cuda.empty_cache()
        print(f"CAE nivel {pct}% evaluado")
    return sin, con, margen           # devolvemos las 3 listas 

def leer_filas_cae_otsu(dir_parcelas, niveles=NIVELES_CMP, n_rep=N_REP_CMP):
    sin, con, margen = [], [], []
    for pct in niveles:
        for rep in range(n_rep):
            ruta = os.path.join(dir_parcelas, f"cae_otsu_{pct:02d}pct_rep_{rep:02d}.npz")
            if not os.path.exists(ruta):
                return None
            z = np.load(ruta, allow_pickle=True)
            n = len(z["upid"])
            sin.append(   _fila_metricas(z["orig"], z["inf"], z["final"], z["relabel"],        pct, rep, n))
            con.append(   _fila_metricas(z["orig"], z["inf"], z["final"], z["relabel_otsu"],   pct, rep, n))
            margen.append(_fila_metricas(z["orig"], z["inf"], z["final"], z["relabel_margen"], pct, rep, n))
    return sin, con, margen


# ==========================================================
# 4. RF: sin y con Otsu a partir de prob_media
# ==========================================================
def filas_rf_otsu_desde_npz(dir_base, niveles=NIVELES_CMP, n_rep=N_REP_CMP):
    """Estructura: dir_base/ruido_XXpct/rep_XX/resultado_parcelas.npz (con prob_media y clases)."""
    sin, con = [], []
    sin_prob = 0
    for pct in niveles:
        for rep in range(n_rep):
            ruta = os.path.join(dir_base, f"ruido_{pct:02d}pct", f"rep_{rep:02d}",
                                "resultado_parcelas.npz")
            if not os.path.exists(ruta):
                print(f"[RF {pct}% rep {rep}] no existe")
                continue
            z = np.load(ruta, allow_pickle=True)
            if "prob_media" not in z.files:
                sin_prob += 1
                continue
            rel_otsu, tau = aplicar_otsu(1.0 - z["prob_media"], z["clases"],
                                         z["inf"], z["final"], z["relabel"])
            f_sin, f_con = _filas_sin_con(z["orig"], z["inf"], z["final"], z["relabel"],
                                          rel_otsu, tau, len(z["clases"]), pct, rep)
            sin.append(f_sin)
            con.append(f_con)
    if sin_prob:
        print(f"\nAVISO: {sin_prob} réplicas del RF no tienen `prob_media`.")
        print("Ejecuta ejecutar() de la versión nueva de farmsar_rf_baseline.py "
              "(solo recalcula las que no la tienen) y vuelve a lanzar esto.")
    return sin, con


# ==========================================================
# 5. COMPROBACIÓN, TABLAS Y CSV
# ==========================================================
def verificar_mismas_parcelas(dir_cae, dir_rf, niveles=NIVELES_CMP, n_rep=N_REP_CMP):
    """CAE y RF deben evaluar las mismas parcelas con las mismas etiquetas infectadas."""
    n_ok = n_mal = n_falta = 0
    for pct in niveles:
        for rep in range(n_rep):
            rc = os.path.join(dir_cae, f"cae_otsu_{pct:02d}pct_rep_{rep:02d}.npz")
            rr = os.path.join(dir_rf, f"ruido_{pct:02d}pct", f"rep_{rep:02d}", "resultado_parcelas.npz")
            if not (os.path.exists(rc) and os.path.exists(rr)):
                n_falta += 1
                continue
            zc, zr = np.load(rc, allow_pickle=True), np.load(rr, allow_pickle=True)
            igual = (len(zc["upid"]) == len(zr["upid"])
                     and np.array_equal(zc["upid"], zr["upid"])
                     and np.array_equal(zc["inf"].astype(str), zr["inf"].astype(str))
                     and np.array_equal(zc["orig"].astype(str), zr["orig"].astype(str)))
            if igual:
                n_ok += 1
            else:
                n_mal += 1
                print(f"  DIFERENTE: {pct}% rep {rep} (CAE {len(zc['upid'])} parcelas, "
                      f"RF {len(zr['upid'])})")
    print(f"\nVerificación CAE vs RF: {n_ok} réplicas idénticas, {n_mal} distintas, "
          f"{n_falta} sin comparar")
    return n_mal == 0


def _ms(filas, pct, clave):
    v = np.array([f[clave] for f in filas if f["pct"] == pct], dtype=float)
    return (v.mean(), v.std()) if len(v) else (np.nan, np.nan)


def _fmt_ref(r):
    if not r:
        return "--"
    a = "--" if np.isnan(r[0]) else f"{r[0]:.2f}"
    b = "--" if np.isnan(r[1]) else f"{r[1]:.2f}"
    return f"{a}/{b}"


def tabla_metodo(nombre, sin, con, ref_sin=None, ref_con=None):
    ref_sin, ref_con = ref_sin or {}, ref_con or {}
    print(f"\n{nombre}")
    print(f"{'ruido':>5} | {'--- sin Otsu ---':^31} | {'--- con Otsu ---':^31} | "
          f"{'art. sin':>9} {'art. con':>9}")
    print(f"{'':>5} | {'recall':>11} {'precisión':>11} {'relab':>6} | "
          f"{'recall':>11} {'precisión':>11} {'relab':>6} | {'r/p':>9} {'r/p':>9}")
    print("-" * 100)
    for pct in sorted({f["pct"] for f in sin + con}):
        cols = []
        for filas in (sin, con):
            rm, rs = _ms(filas, pct, "recall")
            pm, ps = _ms(filas, pct, "precision")
            nm, _ = _ms(filas, pct, "relabels")
            cols.append(f"{rm:.2f}±{rs:.2f} {pm:.2f}±{ps:.2f} {nm:>6.0f}")
        print(f"{pct:>4}% | {cols[0]:>31} | {cols[1]:>31} | "
              f"{_fmt_ref(ref_sin.get(pct)):>9} {_fmt_ref(ref_con.get(pct)):>9}")


def guardar_csv_global(listas, ruta):
    """listas: {(metodo, otsu): filas}"""
    filas_all = []
    for (metodo, otsu), filas in listas.items():
        for f in filas:
            filas_all.append({"metodo": metodo, "otsu": otsu, **f})
    if not filas_all:
        return
    campos = []
    for f in filas_all:
        for k in f:
            if k not in campos:
                campos.append(k)
    with open(ruta, "w", newline="") as fh:
        w = csv.DictWriter(fh, fieldnames=campos)
        w.writeheader()
        w.writerows(filas_all)
    print(f"Guardado: {ruta}")


# ==========================================================
# 6. EJECUCIÓN
# ==========================================================
def tabla_tres_variantes(nombre_cae, sin, con, margen, rf_sin=None, rf_con=None):
    """
    Muestra todos los métodos lado a lado para cada nivel de ruido.
    Muestra tanto recall promediado (μ) como recall agregado (∑).
    """
    def stats(filas,pct):
        if not filas:
            return (float("nan"),) * 6
        fs = [f for f in filas if f["pct"] == pct]
        if not fs:
            return (float("nan"),) * 6
        
        # Recall agregado (∑) — única versión
        ok_total         = sum(f["acierto"]        for f in fs)
        infectadas_total = sum(f["infectadas"]      for f in fs)
        relabels_total   = sum(f["relabels"]        for f in fs)
        fp_total         = sum(f["falso_positivo"]  for f in fs)
        nd_total         = sum(f["no_detectada"]    for f in fs)
        rec_sum  = ok_total / max(infectadas_total, 1)
        
        # Precisión promediada (μ) — media de las 10 réplicas
        prec_mu, _ = _ms(filas, pct, "precision")
        
        # Precisión agregada (∑) — ok_total / relabels_total
        prec_sum = ok_total / max(relabels_total, 1)
        return rec_sum, prec_mu, prec_sum, ok_total, fp_total, nd_total
        
    def celda(filas,pct):
        rec_sum, prec_mu, prec_sum, ok, fp, nd = stats(filas,pct)
        if rec_sum != rec_sum:   # nan check
            return f"{'---':^35}"
        return (
                f"{rec_sum:>6.2f} {prec_mu:>7.2f} {prec_sum:>7.2f} "
                f"{int(ok):>4} {int(fp):>4} {int(nd):>5}"
        )      


    print(f"\n{'=' * 200}")
    print(f" Comparativa completa — todos los métodos")
    print(f"{'=' * 200}")
    print(
        f"{'ruido':>5} | "
        f"{'CAE sin umbral':^37} | "
        f"{'Art. CAE':^9} | "
        f"{'RF sin umbral':^37} | "
        f"{'Art. RF':^9} | "
        f"{'CAE Otsu':^37} | "
        f"{'Art. Otsu':^9} | "
        f"{'RF Otsu':^37} | "
        f"{'CAE Margen MSE':^37}"
    )
    print(
        f"{'':>5} | "
        f"{'rec(∑)':>6} {'prec(μ)':>7} {'prec(∑)':>7} {'ok':>4} {'FP':>4} {'noDet':>5} | "
        f"{'r/p':^9} | "
        f"{'rec(∑)':>6} {'prec(μ)':>7} {'prec(∑)':>7} {'ok':>4} {'FP':>4} {'noDet':>5} | "
        f"{'r/p':^9} | "
        f"{'rec(∑)':>6} {'prec(μ)':>7} {'prec(∑)':>7} {'ok':>4} {'FP':>4} {'noDet':>5} | "
        f"{'r/p':^9} | "
        f"{'rec(∑)':>6} {'prec(μ)':>7} {'prec(∑)':>7} {'ok':>4} {'FP':>4} {'noDet':>5} | "
        f"{'rec(∑)':>6} {'prec(μ)':>7} {'prec(∑)':>7} {'ok':>4} {'FP':>4} {'noDet':>5}"
    )
    print("-" * 200)

    niveles = sorted({f["pct"] for f in sin})

    for pct in niveles:
        pct_int = int(pct)
        ref_cae      = REF_ARTICULO_CAE.get(pct_int,      (float("nan"), float("nan")))
        ref_rf       = REF_ARTICULO_RF.get(pct_int,        (float("nan"), float("nan")))
        ref_cae_otsu = REF_ARTICULO_CAE_OTSU.get(pct_int,  (float("nan"), float("nan")))
        
        
        print(
            f"{pct:>4}% | "                   # ← columna del nivel de ruido
            f"{celda(sin, pct)} | "
            f"{_fmt_ref(ref_cae):^9} | "      # ← referencia CAE sin Otsu
            f"{celda(rf_sin, pct) if rf_sin else '---':^35} | "
            f"{_fmt_ref(ref_rf):^9} | "
            f"{celda(con, pct)} | "
            f"{_fmt_ref(ref_cae_otsu):^9} | "
            f"{celda(rf_con, pct) if rf_con else '---':^35} | "
            f"{celda(margen, pct)}"
        )
     
def ejecutar_comparacion(recalcular_cae=False):
    _comprobar_dependencias()

    res = None if recalcular_cae else leer_filas_cae_otsu(dir_parcelas_cae)
    if res is None:
        print("Calculando CAE (sin umbral, Otsu y margen MSE)...")
        cae_sin, cae_con, cae_margen = evaluar_todos_cae_otsu(dir_parcelas=dir_parcelas_cae)
    else:
        print("CAE: resultados leídos de los .npz ya guardados.")
        cae_sin, cae_con, cae_margen = res   # ahora desempaqueta 3 

    rf_sin, rf_con = filas_rf_otsu_desde_npz(dir_parcelas_rf)
    print(f"Filas CAE: {len(cae_sin)} | filas RF: {len(rf_sin)} "
          f"(esperadas: {len(NIVELES_CMP) * N_REP_CMP})")

    if rf_sin:
        verificar_mismas_parcelas(dir_parcelas_cae, dir_parcelas_rf)

    # Tabla CAE con las 3 variantes
    tabla_tres_variantes(
        "CAE",
        sin    = cae_sin,
        con    = cae_con,
        margen = cae_margen,
        rf_sin = rf_sin,
        rf_con = rf_con
    )   

    guardar_csv_global({
        ("CAE", "sin"):    cae_sin,
        ("CAE", "otsu"):   cae_con,
        ("CAE", "margen"): cae_margen,   # nueva entrada 
        ("RF",  "sin"):    rf_sin,
        ("RF",  "con"):    rf_con,
    }, ruta_csv_salida)

    return {
        "cae_sin":    cae_sin,
        "cae_con":    cae_con,
        "cae_margen": cae_margen,
        "rf_sin":     rf_sin,
        "rf_con":     rf_con,
    }


resultados = ejecutar_comparacion()

Calculando CAE (sin umbral, Otsu y margen MSE)...
CAE nivel 1% evaluado
Sin modelo para Fallow: /home/angelo/MyStore/Proyecto Investigador/2017/entrenamiento train fiables test/modelos_experimento_farmsar_test_v1/ruido_05pct/rep_01/cae_pasada_10_crop_Fallow.pth
Sin modelo para Fallow: /home/angelo/MyStore/Proyecto Investigador/2017/entrenamiento train fiables test/modelos_experimento_farmsar_test_v1/ruido_05pct/rep_02/cae_pasada_10_crop_Fallow.pth
Sin modelo para Fallow: /home/angelo/MyStore/Proyecto Investigador/2017/entrenamiento train fiables test/modelos_experimento_farmsar_test_v1/ruido_05pct/rep_05/cae_pasada_10_crop_Fallow.pth
Sin modelo para Fallow: /home/angelo/MyStore/Proyecto Investigador/2017/entrenamiento train fiables test/modelos_experimento_farmsar_test_v1/ruido_05pct/rep_07/cae_pasada_10_crop_Fallow.pth
CAE nivel 5% evaluado
Sin modelo para Fallow: /home/angelo/MyStore/Proyecto Investigador/2017/entrenamiento train fiables test/modelos_experimento_farmsar_test_v1/ruido

In [38]:
# ============================================================
# TABLAS SEPARADAS POR MÉTODO
# ============================================================
def aplicar_margen_prob(prob_media, clases, etiq_inf, final, relabel,
                        margen_base=0.20, margen_2do=0.30):
    col = {c: j for j, c in enumerate(clases)}
    rel = relabel.copy()

    for i in np.flatnonzero(relabel):

        clase_original = etiq_inf[i]

        if clase_original not in col:
            rel[i] = False
            continue

        probs = prob_media[i]

        # Ordenar de mayor a menor probabilidad
        orden_probs     = np.argsort(probs)[::-1]
        prob_candidata  = probs[orden_probs[0]]
        clase_segunda   = clases[orden_probs[1]]   # 2ª mayor prob

        prob_original = probs[col[clase_original]]

        if prob_original <= 0:
            rel[i] = False
            continue

        diferencia = (prob_candidata - prob_original) / prob_original

        # Caso ambiguo: clase original es la 2ª mejor → más conservador
        if clase_segunda == clase_original:
            rel[i] = diferencia > margen_2do
        else:
            rel[i] = diferencia > margen_base

    return rel

def filas_rf_margen_desde_npz(dir_base, niveles=NIVELES_CMP, n_rep=N_REP_CMP,
                               margen_base=0.20, margen_2do=0.30):
    """Aplica margen relativo de probabilidad sobre los .npz del RF."""
    filas = []
    sin_prob = 0

    for pct in niveles:
        for rep in range(n_rep):
            ruta = os.path.join(dir_base, f"ruido_{pct:02d}pct", f"rep_{rep:02d}",
                                "resultado_parcelas.npz")
            if not os.path.exists(ruta):
                print(f"[RF margen {pct}% rep {rep}] no existe")
                continue
            z = np.load(ruta, allow_pickle=True)
            if "prob_media" not in z.files:
                sin_prob += 1
                continue

            rel_margen = aplicar_margen_prob(
                z["prob_media"], z["clases"],
                z["inf"], z["final"], z["relabel"],
                margen_base=margen_base, margen_2do=margen_2do
            )

            filas.append(_fila_metricas(
                z["orig"], z["inf"], z["final"], rel_margen, pct, rep, len(z["upid"])
            ))

    if sin_prob:
        print(f"\nAVISO: {sin_prob} réplicas sin `prob_media`.")
    return filas

def _imprimir_tabla(nombre, filas_tuyas, filas_rf, ref_tuyas, ref_rf):
    """Imprime una tabla para un método concreto (sin umbral, Otsu o Margen MSE)."""

    print(f"\n{'=' * 95}")
    print(f" {nombre}")
    print(f"{'=' * 95}")
    print(
        f"{'ruido':>5} | "
        f"{'CAE':^37} | "
        f"{'Art. CAE':^9} | "
        f"{'RF':^37} | "
        f"{'Art. RF':^9}"
    )
    print(
        f"{'':>5} | "
        f"{'rec(∑)':>6} {'prec(μ)':>7} {'prec(∑)':>7} {'ok':>4} {'FP':>4} {'noDet':>5} | "
        f"{'r/p':^9} | "
        f"{'rec(∑)':>6} {'prec(μ)':>7} {'prec(∑)':>7} {'ok':>4} {'FP':>4} {'noDet':>5} | "
        f"{'r/p':^9}"
    )
    print("-" * 95)

    niveles = sorted({f["pct"] for f in filas_tuyas})

    for pct in niveles:

        pct_int      = int(pct)
        ref_cae_val  = ref_tuyas.get(pct_int, (float("nan"), float("nan")))
        ref_rf_val   = ref_rf.get(pct_int,    (float("nan"), float("nan")))

        def stats(filas,pct):
            if not filas:
                return (float("nan"),) * 6
            fs = [f for f in filas if f["pct"] == pct]
            if not fs:
                return (float("nan"),) * 6

            ok_total         = sum(f["acierto"]       for f in fs)
            infectadas_total = sum(f["infectadas"]     for f in fs)
            relabels_total   = sum(f["relabels"]       for f in fs)
            fp_total         = sum(f["falso_positivo"] for f in fs)
            nd_total         = sum(f["no_detectada"]   for f in fs)

            rec_sum  = ok_total / max(infectadas_total, 1)
            prec_mu, _ = _ms(filas, pct, "precision")
            prec_sum = ok_total / max(relabels_total, 1)

            return rec_sum, prec_mu, prec_sum, ok_total, fp_total, nd_total

        def celda(filas,pct):
            rec_sum, prec_mu, prec_sum, ok, fp, nd = stats(filas,pct)
            if rec_sum != rec_sum:
                return f"{'---':^37}"
            return (
                f"{rec_sum:>6.2f} {prec_mu:>7.2f} {prec_sum:>7.2f} "
                f"{int(ok):>4} {int(fp):>4} {int(nd):>5}"
            )

        print(
            f"{pct:>4}% | "
            f"{celda(filas_tuyas,pct)} | "
            f"{_fmt_ref(ref_cae_val):^9} | "
            f"{celda(filas_rf,pct) if filas_rf else '---':^37} | "
            f"{_fmt_ref(ref_rf_val):^9}"
        )       
_imprimir_tabla(
    "SIN UMBRAL — solo regla del 75%",
    filas_tuyas = resultados["cae_sin"],
    filas_rf    = resultados["rf_sin"],
    ref_tuyas   = REF_ARTICULO_CAE,
    ref_rf      = REF_ARTICULO_RF,
)

_imprimir_tabla(
    "OTSU — doble condición (paper)",
    filas_tuyas = resultados["cae_con"],
    filas_rf    = resultados["rf_con"],
    ref_tuyas   = REF_ARTICULO_CAE_OTSU,
    ref_rf      = {},
)

rf_margen = filas_rf_margen_desde_npz(dir_parcelas_rf)

_imprimir_tabla(
    "MARGEN MSE (CAE) / MARGEN PROB (RF) — propuesta propia",
    filas_tuyas = resultados["cae_margen"],
    filas_rf    = rf_margen,
    ref_tuyas   = {},
    ref_rf      = {},
)


 SIN UMBRAL — solo regla del 75%
ruido |                  CAE                  | Art. CAE  |                  RF                   |  Art. RF 
      | rec(∑) prec(μ) prec(∑)   ok   FP noDet |    r/p    | rec(∑) prec(μ) prec(∑)   ok   FP noDet |    r/p   
-----------------------------------------------------------------------------------------------
   1% |   0.91    0.52    0.42   93  124     7 | 0.92/0.71 |   0.98    0.67    0.67  100   49     2 | 0.95/0.56
   5% |   0.89    0.72    0.71  527  199    46 | 0.90/0.90 |   0.98    0.91    0.91  576   56    14 | 0.94/0.86
  10% |   0.92    0.85    0.84 1117  189    76 | 0.88/0.92 |   0.96    0.94    0.94 1170   74    45 | 0.90/0.91
  15% |   0.89    0.81    0.80 1607  329   136 | 0.81/0.84 |   0.96    0.94    0.94  524   32    19 | 0.87/0.90
  20% |   0.90    0.87    0.86 2160  273   172 | 0.71/0.77 |                  ---                  | 0.83/0.89
  25% |   0.92    0.94    0.94  557   25    43 | 0.67/0.73 |                  ---        